# 2026 COMP90042 Project — Group 073 (CNN+BiLSTM+Multihead, balanced)
*Fact-checking: FAISS dense retrieval + multi-kernel CNN + BiLSTM + multi-head attention classifier with class-balanced loss.*

# Readme

**Before running:**

1. From your local machine, push the working branch to GitHub:
   `git push -u origin <branch>`
2. Add a GitHub Personal Access Token to Colab Secrets (only needed for private repos).
3. On Google Drive at `/content/drive/MyDrive/COMP90042_2026/`, place the data:
   - Required: `data/evidence.json` (~1 GB), `data/train-claims.json`, `data/dev-claims.json`, `data/test-claims-unlabelled.json`
   - Optional (saves ~1 hr): `cache/faiss/*.faiss` and `cache/faiss/*_evidence_ids.json`

Cell 1.1 clones the code from GitHub to `/content/COMP90042_2026` (Colab's fast local SSD) and symlinks `data/`, `cache/`, `checkpoints/`, `outputs/` from Drive — so code is git-managed and data persists across sessions.

**Pipeline:** FAISS dense retrieval (1.2M passages) → CNN+BiLSTM+Multi-head Attention classifier trained with class-balanced cross-entropy.

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 1.1 · Setup — Sync code from GitHub, mount Drive, install packages

import os

# Stop JAX/TF from preallocating GPU memory.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import shutil
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

# -- EDIT IF NEEDED ----------------------------------------------------------
GITHUB_USER = "VitaChien"
REPO_NAME = "COMP90042_2026"
BRANCH = "reseviv"
DRIVE_DATA = "/content/drive/MyDrive/COMP90042_2026"
PROJECT_ROOT = "/content/COMP90042_2026"
# ----------------------------------------------------------------------------

repo_url = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"
if not os.path.exists(PROJECT_ROOT):
    subprocess.check_call(["git", "clone", "-b", BRANCH, repo_url, PROJECT_ROOT])
else:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull"])

for sub in ("data", "cache", "checkpoints", "outputs"):
    src = f"{DRIVE_DATA}/{sub}"
    dst = f"{PROJECT_ROOT}/{sub}"
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst) and not os.path.islink(dst):
        shutil.rmtree(dst)
    if not os.path.islink(dst):
        os.symlink(src, dst)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "transformers>=4.40",
        "sentence-transformers>=2.7",
        "faiss-cpu",
        "scikit-learn>=1.3",
        "tqdm>=4.65",
    ]
)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# @title 1.2 · Verify data files exist
from pathlib import Path

root = Path(PROJECT_ROOT)
checks = [
    ("data/evidence.json", "required (~1 GB)"),
    ("data/train-claims.json", "required"),
    ("data/dev-claims.json", "required"),
    ("data/test-claims-unlabelled.json", "required"),
    ("cache/faiss", "optional — saves ~1 hr"),
]
missing_required = False
for rel, note in checks:
    p = root / rel
    if p.exists():
        size = f"{p.stat().st_size/1e6:.0f} MB" if p.is_file() else "dir"
        print(f"  OK  {rel:45s} {size:10s}  ({note})")
    else:
        icon = "XX" if "required" in note else "--"
        print(f'  {icon}  {rel:45s} {"MISSING":10s}  ({note})')
        if "required" in note:
            missing_required = True

if missing_required:
    raise FileNotFoundError("Required data files are missing. Upload them to Drive first.")

In [ ]:
# @title 1.3 · Imports, utilities, load data, build vocab, build/load FAISS index

import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

from src.v3_helpers import build_vocab_full_corpus, simple_tokenise

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# ---------------- reproducibility ----------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed(42)


# ---------------- paths ----------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
CACHE_DIR = Path("cache")
OUTPUT_DIR.mkdir(exist_ok=True)
(CACHE_DIR / "faiss").mkdir(parents=True, exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"


# ---------------- utilities ----------------
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def get_claim_items(claims_json: Dict) -> List[Tuple[str, Dict]]:
    return list(claims_json.items())


def concatenate_evidence(
    evidence_ids: List[str],
    evidence_corpus: Dict[str, str],
    max_evidence: int = 5,
) -> str:
    selected_ids = evidence_ids[:max_evidence]
    evidence_texts = []
    for eid in selected_ids:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])
    if len(evidence_texts) == 0:
        return "No relevant evidence found."
    return " ".join(evidence_texts)


# ---------------- load data ----------------
evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Number of evidence passages:", len(evidence_corpus))
print("Number of train claims:", len(train_claims))
print("Number of dev claims:", len(dev_claims))


# ---------------- vocab ----------------
vocab = build_vocab_full_corpus(train_claims, evidence_corpus, min_freq=2, max_vocab_size=50_000)
print("Vocab size:", len(vocab))


# ---------------- FAISS retriever (build if missing, else load) ----------------
import faiss
from sentence_transformers import SentenceTransformer


class CachedFAISSDenseRetriever:
    def __init__(
        self,
        evidence_corpus,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        cache_dir="cache/faiss",
        batch_size=64,
        device=None,
        force_rebuild=False,
    ):
        self.evidence_corpus = evidence_corpus
        self.evidence_ids = list(evidence_corpus.keys())
        self.evidence_texts = [evidence_corpus[eid] for eid in self.evidence_ids]

        self.model_name = model_name
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        safe_model_name = model_name.replace("/", "_")
        self.index_path = self.cache_dir / f"{safe_model_name}.faiss"
        self.ids_path = self.cache_dir / f"{safe_model_name}_evidence_ids.json"

        self.model = SentenceTransformer(model_name, device=device)

        if (
            self.index_path.exists()
            and self.ids_path.exists()
            and not force_rebuild
        ):
            print("Loading cached FAISS index...")
            with open(self.ids_path, "r", encoding="utf-8") as f:
                cached_ids = json.load(f)
            if cached_ids != self.evidence_ids:
                print("Cached IDs mismatch — rebuilding.")
                self._build_and_save_index(batch_size)
            else:
                self.index = faiss.read_index(str(self.index_path))
                print("Cached FAISS index loaded.")
        else:
            print("No valid FAISS cache found. Building index...")
            self._build_and_save_index(batch_size)

    def _build_and_save_index(self, batch_size):
        evidence_embeddings = self.model.encode(
            self.evidence_texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        embedding_dim = evidence_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(embedding_dim)
        self.index.add(evidence_embeddings)
        faiss.write_index(self.index, str(self.index_path))
        with open(self.ids_path, "w", encoding="utf-8") as f:
            json.dump(self.evidence_ids, f, indent=2)
        print("Saved FAISS index to:", self.index_path)

    def retrieve(self, claim_text, top_k=5):
        claim_embedding = self.model.encode(
            [claim_text],
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        scores, indices = self.index.search(claim_embedding, top_k)
        return [self.evidence_ids[i] for i in indices[0]]


retriever = CachedFAISSDenseRetriever(
    evidence_corpus=evidence_corpus,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_dir="cache/faiss",
    batch_size=64,
    device=device,
    force_rebuild=False,
)

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 2.1 · Encode text + CNNBiLSTMDataset + collate fn

def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"]
        + simple_tokenise(claim_text)
        + ["<EVIDENCE>"]
        + simple_tokenise(evidence_text)
    )
    return encode_tokens(tokens, vocab, max_len=max_len)


class CNNBiLSTMDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        max_len=512,
        max_evidence=5,
        use_gold_evidence=True,
        retriever=None,
        retrieval_top_k=5,
        is_test=False,
    ):
        self.items = list(claims_json.items())
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.max_len = max_len
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            evidence_ids = self.retriever.retrieve(
                claim_text, top_k=self.retrieval_top_k
            )

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence,
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len,
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids,
        }

        if not self.is_test:
            item["label"] = torch.tensor(
                LABEL2ID[instance["claim_label"]], dtype=torch.long
            )

        return item


def cnn_bilstm_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(
        input_ids, batch_first=True, padding_value=vocab["<PAD>"]
    )
    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch],
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output

In [ ]:
# @title 2.2 · Model — multi-kernel CNN + BiLSTM + Multi-head Attention + Attention Pooling

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attention(hidden_states).squeeze(-1)
        if attention_mask is not None:
            attention_mask = attention_mask.bool()
            scores = scores.masked_fill(~attention_mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)
        return pooled


class CNNBiLSTMMultiheadClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=0,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx,
        )
        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList(
            [
                nn.Conv1d(
                    in_channels=embedding_dim,
                    out_channels=cnn_channels,
                    kernel_size=k,
                    padding=k // 2,
                )
                for k in kernel_sizes
            ]
        )

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        lstm_output_dim = lstm_hidden_dim * 2

        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=lstm_output_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True,
        )
        self.layer_norm = nn.LayerNorm(lstm_output_dim)

        self.attention_pooling = AttentionPooling(lstm_output_dim)
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels),
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        conv_input = embedded.transpose(1, 2)
        conv_outputs = []
        for conv in self.convs:
            x = F.relu(conv(conv_input))
            conv_outputs.append(x)
        conv_output = torch.cat(conv_outputs, dim=1)

        lstm_input = conv_output.transpose(1, 2)
        lstm_output, _ = self.bilstm(lstm_input)

        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0

        attn_output, _ = self.multihead_attn(
            query=lstm_output,
            key=lstm_output,
            value=lstm_output,
            key_padding_mask=key_padding_mask,
        )
        attn_output = self.layer_norm(attn_output + lstm_output)

        pooled_output = self.attention_pooling(
            attn_output, attention_mask=attention_mask
        )
        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)
        return logits

In [ ]:
# @title 2.3 · Evaluate fn + class weights + train fn (class-balanced cross-entropy)

def evaluate_cnn_bilstm(model, dataloader, device="cpu"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).long()

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    per_class_f1 = f1_score(
        all_labels,
        all_preds,
        average=None,
        labels=list(range(4)),
        zero_division=0,
    )

    print("Dev classification accuracy:", round(acc, 4))
    print("Dev macro F1:", round(macro_f1, 4))
    print("Dev weighted F1:", round(weighted_f1, 4))

    print("\nPer-class F1:")
    for i, score in enumerate(per_class_f1):
        print(f"{ID2LABEL[i]}: {score:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            all_labels,
            all_preds,
            labels=list(range(4)),
            target_names=[ID2LABEL[i] for i in range(4)],
            zero_division=0,
        )
    )

    print("\nConfusion matrix:")
    print(confusion_matrix(all_labels, all_preds, labels=list(range(4))))

    return acc, macro_f1, weighted_f1


def get_class_weights_from_dataset(dataset, num_labels, device):
    labels = []
    for item in dataset:
        label = item["label"] if "label" in item else item["labels"]
        labels.append(int(label))
    labels = np.array(labels)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_labels),
        y=labels,
    )
    return torch.tensor(class_weights, dtype=torch.float).to(device)


def train_cnn_bilstm_multikernel_multihead_balanced(
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device="cpu",
):
    set_seed(42)

    train_dataset = CNNBiLSTMDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False,
    )

    dev_dataset = CNNBiLSTMDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False,
    )

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=cnn_bilstm_collate_fn,
        generator=g,
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn,
    )

    class_weights = get_class_weights_from_dataset(
        dataset=train_dataset, num_labels=4, device=device
    )

    model = CNNBiLSTMMultiheadClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=vocab["<PAD>"],
    ).to(device)

    optimiser = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-4
    )

    print("Class weights:", class_weights)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_macro_f1 = 0.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimiser.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print("Training loss:", round(avg_loss, 4))

        dev_acc, dev_macro_f1, dev_weighted_f1 = evaluate_cnn_bilstm(
            model, dev_loader, device
        )

        print("Dev accuracy with gold evidence:", round(dev_acc, 4))
        print("Dev macro F1 with gold evidence:", round(dev_macro_f1, 4))
        print("Dev weighted F1 with gold evidence:", round(dev_weighted_f1, 4))

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            print("New best model saved in memory.")

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model

In [ ]:
# @title 2.4 · Run training (10 epochs, class-balanced loss)

cnn_bilstm_multihead_model = train_cnn_bilstm_multikernel_multihead_balanced(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    max_len=256,
    max_evidence=4,
    device=device,
)

# Save best weights
ckpt_path = Path("checkpoints") / "cnn_bilstm_multihead_balanced.pt"
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(cnn_bilstm_multihead_model.state_dict(), ckpt_path)
print("Saved checkpoint:", ckpt_path)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 3.1 · Predict on dev set (FAISS top-10 -> use top-4)

def predict_claims_cnn_bilstm(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device="cpu",
):
    dataset = CNNBiLSTMDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=cnn_bilstm_collate_fn,
    )

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(
                batch["claim_ids"], pred_ids, batch["evidence_ids"]
            ):
                predictions[claim_id] = {
                    "claim_text": claims_json[claim_id]["claim_text"],
                    "claim_label": ID2LABEL[pred_id],
                    "evidences": evidence_ids[:max_evidence],
                }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)
    return predictions


DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm_multihead_bal_4evi.json"

dev_predictions_cnn_bilstm = predict_claims_cnn_bilstm(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_multihead_model,
    vocab=vocab,
    output_path=DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=False,
    device=device,
)

In [ ]:
# @title 3.2 · Evaluate dev predictions — classification F1 + eval.py

def evaluate_dev_predictions_classification_f1(dev_claims, predictions):
    gold_labels = []
    pred_labels = []

    for claim_id, gold_instance in dev_claims.items():
        if claim_id not in predictions:
            continue
        gold_labels.append(LABEL2ID[gold_instance["claim_label"]])
        pred_labels.append(LABEL2ID[predictions[claim_id]["claim_label"]])

    acc = accuracy_score(gold_labels, pred_labels)
    macro_f1 = f1_score(
        gold_labels, pred_labels, average="macro", zero_division=0
    )
    weighted_f1 = f1_score(
        gold_labels, pred_labels, average="weighted", zero_division=0
    )
    per_class_f1 = f1_score(
        gold_labels,
        pred_labels,
        average=None,
        labels=list(range(4)),
        zero_division=0,
    )

    print("Dev classification accuracy:", round(acc, 4))
    print("Dev classification macro F1:", round(macro_f1, 4))
    print("Dev classification weighted F1:", round(weighted_f1, 4))

    print("\nPer-class F1:")
    for i, score in enumerate(per_class_f1):
        print(f"{ID2LABEL[i]}: {score:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            gold_labels,
            pred_labels,
            labels=list(range(4)),
            target_names=[ID2LABEL[i] for i in range(4)],
            zero_division=0,
        )
    )

    print("\nConfusion matrix:")
    print(confusion_matrix(gold_labels, pred_labels, labels=list(range(4))))

    return acc, macro_f1, weighted_f1


dev_cls_acc, dev_cls_macro_f1, dev_cls_weighted_f1 = (
    evaluate_dev_predictions_classification_f1(
        dev_claims=dev_claims, predictions=dev_predictions_cnn_bilstm
    )
)

# Run official eval.py for Evidence F-score + Harmonic Mean
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        f"{PROJECT_ROOT}/eval.py",
        "--predictions",
        str(DEV_PRED_PATH),
        "--groundtruth",
        str(DEV_PATH),
    ]
)

In [ ]:
# @title 3.3 · Generate final predictions on test set (for submission)

test_claims = load_json(TEST_PATH)

TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_cnn_bilstm_multihead_bal.json"

test_predictions = predict_claims_cnn_bilstm(
    claims_json=test_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model=cnn_bilstm_multihead_model,
    vocab=vocab,
    output_path=TEST_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=4,
    batch_size=32,
    max_len=256,
    is_test=True,
    device=device,
)

print(f"\nTest predictions saved to {TEST_PRED_PATH}")

## Object Oriented Programming codes here

All OOP code is inline above:

| Class | Section | Purpose |
|-------|---------|---------|
| `CachedFAISSDenseRetriever` | 1.3 | Dense retriever — builds/loads FAISS index over evidence corpus |
| `CNNBiLSTMDataset` | 2.1 | Pairs claim with gold (train) or retrieved (predict) evidence |
| `AttentionPooling` | 2.2 | Learnable weighted-sum pooling over sequence |
| `CNNBiLSTMMultiheadClassifier` | 2.2 | Multi-kernel CNN → BiLSTM → Multi-head Attention → AttentionPooling → MLP |
